# Controlled posterior behavioral gate with Qwen3.5-4B

This experiment separates posterior computation from prompt and response-label confounds. It contains an explicit comparison control, an N=2/N=4 factorial ladder, and a fixed-transcript N=8 generalization benchmark. Every reliability replay holds its observable evidence fixed. Candidate and partition orientations are paired, and all six A/B/C semantic mappings are scored.

The primary continuous measurement is the model's counterbalanced **behavioral decision logit**, compared against exact posterior log-odds. It is not described as an elicited posterior probability. Methodological sources and their use are documented in [`CITATIONS.md`](../CITATIONS.md), especially `ZhangNanda2024`, `GeigerEtAl2025`, and `LieberumEtAl2023`.

Every scored evaluation is exactly one user-role message containing the rules, observations, probe, and response instruction. **Prompt-format revision:** rerun the model-evaluation cells before interpreting rendered outputs or replacing saved result artifacts; earlier results used a separate system message and are not directly comparable.

## What this notebook is trying to establish

Notebook 02 asks whether behavior changes when the same transcript is replayed with a different reliability. This notebook goes further: it asks **where posterior reasoning first breaks as the required computation becomes more complex**.

It is organized as a behavioral gate:

1. Check that Qwen understands the answer format and can compare explicit weights.
2. Test one noisy observation with only two candidates.
3. Add more observations while keeping two candidates.
4. Add overlapping evidence over four candidates.
5. Test generalization to eight candidates and both candidate and partition comparisons.

A later stage is only scientifically informative if the earlier stages work. For example, poor `N=8` performance is hard to call a scaling failure if the model already fails to interpret one YES/NO observation at `N=2`. Likewise, a wrong A/B/C token is not automatically a Bayesian-reasoning failure if the model did not follow the requested answer format. The controls below are designed to separate these possibilities.

## Generate the model-independent experiment bank

The default primary grid has five interior reliability values. Set `INCLUDE_ENDPOINTS=1` only for the separately reported r=0 and r=1 diagnostics. Endpoint log-odds may be infinite, and candidate ties can be 0=0.

### The simple-to-complex ladder

| Stage | Candidates | Evidence | What it isolates |
|---|---:|---|---|
| Stage 0: elicitation | Two displayed weights | No narrative evidence | Can Qwen compare numbers, recognize equality, and emit the mapped label? |
| `one_observation` | 2 | One membership answer | Can it apply one noisy-channel likelihood, including inversion below `r=0.5`? |
| `accumulation` | 2 | Two, three, or four answers | Can it combine repeated log-likelihood contributions? |
| `compositional` | 4 | Three overlapping subset questions | Can it track evidence that applies to several candidates at once? |
| `n8` | 8 | Three size-four subset questions | Does the behavior generalize to a larger domain and to total probability over partitions? |

The small stages enumerate every visible YES/NO pattern and repeat them under several randomized candidate-label worlds. The `N=8` stage uses 32 fixed question schedules and all eight three-answer patterns per schedule. Each history is replayed at all five interior reliabilities.

In [4]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt

from mats_experiments.analysis import (
    balanced_accuracy,
    clustered_bootstrap_mean,
    heuristic_accuracies,
    spearman_correlation,
)
from mats_experiments.controlled_posterior import (
    LABEL_ASSIGNMENTS,
    ExperimentConfig,
    N8Config,
    build_elicitation_controls,
    build_ladder_examples,
    build_n8_examples,
    messages_for_elicitation,
    messages_for_probe,
    serialize_example,
    write_jsonl,
)

MODEL_PATH = REPO_ROOT / 'models' / 'Qwen--Qwen3.5-4B'
ARTIFACT_DIR = REPO_ROOT / 'artifacts' / 'controlled_posterior'
INCLUDE_ENDPOINTS = os.environ.get('INCLUDE_ENDPOINTS', '0') == '1'

elicitation_controls = build_elicitation_controls()
ladder_examples = build_ladder_examples(
    ExperimentConfig(include_endpoint_diagnostics=INCLUDE_ENDPOINTS)
)
n8_examples = build_n8_examples(
    N8Config(num_schedules=32), include_endpoint_diagnostics=INCLUDE_ENDPOINTS
)
controlled_examples = [*ladder_examples, *n8_examples]

write_jsonl(ARTIFACT_DIR / 'ladder.jsonl', [serialize_example(row) for row in ladder_examples])
write_jsonl(ARTIFACT_DIR / 'n8_fixed_bank.jsonl', [serialize_example(row) for row in n8_examples])
print(f'{len(elicitation_controls)=}, {len(ladder_examples)=}, {len(n8_examples)=}')
print(f'Primary aggregate probe decisions: {sum(len(row.probes) for row in controlled_examples):,}')

len(elicitation_controls)=6, len(ladder_examples)=1480, len(n8_examples)=1280
Primary aggregate probe decisions: 8,080


### Anatomy of one controlled example

A `bank_id` identifies an observable history and its probe meanings. For that bank, the candidate labels, questions, SOURCE answers, and left/right alternatives are held fixed. An `example_id` adds one reliability value and therefore one exact posterior.

The distinction supports two useful counterfactuals:

- **Reliability replay:** compare the same bank at `r` and `1-r`. For a symmetric noisy channel, every evidence log-likelihood contribution changes sign. A non-tied candidate preference should therefore reverse.
- **Probe orientation:** ask `left versus right` and then `right versus left`. Exact log-odds negate under this display-only reversal. This catches left/right presentation preferences.

`world_id` changes the visible candidate labels while preserving the canonical evidence structure. This prevents a result from depending entirely on special identities such as candidate `1` or the first half of the displayed domain.

The exact targets are computed with rational arithmetic. The model is never shown the posterior, the hidden canonical world, `bank_id`, or the normative choice.

## Design invariants and target audit

These checks establish the intended independent unit and counterfactual controls before model inference. Reliability variants share the same bank ID, observation history, and probe meanings. Each forward probe has a reversed-orientation partner.

In [5]:
from collections import Counter, defaultdict
from fractions import Fraction

variants_by_bank = defaultdict(list)
for example in controlled_examples:
    variants_by_bank[example.bank_id].append(example)
for variants in variants_by_bank.values():
    assert all(item.observations == variants[0].observations for item in variants)
    assert all(
        tuple((probe.kind, probe.left, probe.right) for probe in item.probes)
        == tuple((probe.kind, probe.left, probe.right) for probe in variants[0].probes)
        for item in variants
    )
assert all(
    len(observation.subset) == len(set(observation.subset))
    for example in n8_examples
    for observation in example.observations
)
assert all(
    len(set(example.posterior)) == 1
    for example in controlled_examples
    if example.reliability == Fraction(1, 2)
)

for schedule_id in range(32):
    for reliability in N8Config().reliabilities:
        assert sum(
            example.prior_predictive_probability
            for example in n8_examples
            if example.schedule_id == schedule_id and example.reliability == reliability
        ) == 1

target_counts = Counter(
    (example.stage, float(example.reliability), probe.kind, probe.normative_choice, probe.tie_type)
    for example in controlled_examples
    for probe in example.probes
)
for key, count in sorted(target_counts.items(), key=str):
    print(key, count)

('accumulation', 0.1, 'candidate', 'left', None) 160
('accumulation', 0.1, 'candidate', 'right', None) 160
('accumulation', 0.1, 'candidate', 'tie', 'equal_positive') 128
('accumulation', 0.3, 'candidate', 'left', None) 160
('accumulation', 0.3, 'candidate', 'right', None) 160
('accumulation', 0.3, 'candidate', 'tie', 'equal_positive') 128
('accumulation', 0.5, 'candidate', 'tie', 'uniform_positive') 448
('accumulation', 0.7, 'candidate', 'left', None) 160
('accumulation', 0.7, 'candidate', 'right', None) 160
('accumulation', 0.7, 'candidate', 'tie', 'equal_positive') 128
('accumulation', 0.9, 'candidate', 'left', None) 160
('accumulation', 0.9, 'candidate', 'right', None) 160
('accumulation', 0.9, 'candidate', 'tie', 'equal_positive') 128
('compositional', 0.1, 'candidate', 'left', None) 48
('compositional', 0.1, 'candidate', 'right', None) 48
('compositional', 0.1, 'candidate', 'tie', 'equal_positive') 32
('compositional', 0.3, 'candidate', 'left', None) 48
('compositional', 0.3, 'ca

## What is sent to Qwen, and why it is scored six times

Every evaluation is one user-role message containing: the complete channel rules, an `OBSERVATIONS` block, the TEST probe, three answer options, and `Reply with exactly A, B, or C.` Qwen is positioned only at the assistant response boundary.

The same semantic probe is rendered under all six mappings of `left`, `right`, and `tie` to A/B/C. For example:

| Rendering | A means | B means | C means |
|---|---|---|---|
| One mapping | left | right | tie |
| Another mapping | tie | left | right |

The label probabilities are mapped back to meaning before averaging. This reduces a fixed preference for token A, B, or C. The six renderings are not six independent datapoints; together they form one aggregate semantic decision.

Two score types are deliberately retained:

- **Conditional A/B/C score:** renormalizes the probabilities of A, B, and C and asks which permitted option is strongest.
- **Unrestricted next token:** records the actual full-vocabulary argmax and the total probability mass on A/B/C.

If the unrestricted token is something like `To` and A/B/C mass is tiny, the conditional choice is a forced diagnostic—not the answer Qwen would naturally emit. This is an elicitation failure that must be reported separately from posterior-ranking accuracy.

## Load Qwen and score the counterbalanced grid

This cell is intentionally marked `run-when-ready`. The full primary bank requires roughly 48k prompt forward passes after six-way label counterbalancing. Every evaluation is packaged as one user-role message containing the rules, observations, probe, and response instruction. A deterministic 128-example subset is replayed under alternate answer vocabularies and reliability notations as separately labeled robustness arms.

In [6]:
import torch
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

from mats_experiments.qwen_scoring import (
    aggregate_counterbalanced_scores,
    aggregate_elicitation_scores,
    generate_responses_batch,
    label_token_ids,
    score_labels_batch,
    validate_contextual_label_tokens,
)

assert MODEL_PATH.exists(), f'Missing local checkpoint: {MODEL_PATH}'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
processor.tokenizer.padding_side = 'left'
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_PATH, dtype=MODEL_DTYPE, local_files_only=True
).to(DEVICE)
model.eval()
CHOICE_TOKEN_IDS = label_token_ids(processor)
validate_contextual_label_tokens(processor, CHOICE_TOKEN_IDS, enable_thinking=False)
print(DEVICE, MODEL_DTYPE, CHOICE_TOKEN_IDS)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 31.36 GiB of which 15.06 MiB is free. Process 13031 has 20.59 GiB memory in use. Process 5886 has 9.28 GiB memory in use. Including non-PyTorch memory, this process has 1.46 GiB memory in use. Of the allocated memory 956.97 MiB is allocated by PyTorch, and 37.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
PRIMARY_ARM = {
    'transcript_format': 'user_only',
    'answer_vocabulary': 'yes_no',
    'reliability_format': 'decimal',
    'enable_thinking': False,
}
step = max(1, len(controlled_examples) // 128)
robustness_examples = controlled_examples[::step][:128]
ROBUSTNESS_ARMS = (
    {**PRIMARY_ARM, 'answer_vocabulary': 'true_false'},
    {**PRIMARY_ARM, 'answer_vocabulary': 'symbols'},
    {**PRIMARY_ARM, 'reliability_format': 'percent'},
    {**PRIMARY_ARM, 'reliability_format': 'fraction'},
)
aggregate_jobs = [
    (example, probe, PRIMARY_ARM)
    for example in controlled_examples
    for probe in example.probes
]
aggregate_jobs.extend(
    (example, probe, arm)
    for arm in ROBUSTNESS_ARMS
    for example in robustness_examples
    for probe in example.probes
)

INFERENCE_BATCH_SIZE = int(os.environ.get('INFERENCE_BATCH_SIZE', '48'))
assert INFERENCE_BATCH_SIZE > 0 and INFERENCE_BATCH_SIZE % len(LABEL_ASSIGNMENTS) == 0
AGGREGATES_PER_BATCH = INFERENCE_BATCH_SIZE // len(LABEL_ASSIGNMENTS)
results = []
for batch_start in range(0, len(aggregate_jobs), AGGREGATES_PER_BATCH):
    batch = aggregate_jobs[batch_start:batch_start + AGGREGATES_PER_BATCH]
    messages = [
        messages_for_probe(
            example, probe, assignment,
            answer_vocabulary=arm['answer_vocabulary'],
            reliability_format=arm['reliability_format'],
        )
        for example, probe, arm in batch
        for assignment in LABEL_ASSIGNMENTS
    ]
    flat_scores = score_labels_batch(
        model=model, processor=processor, device=DEVICE, messages_batch=messages,
        choice_token_ids=CHOICE_TOKEN_IDS, enable_thinking=False,
    )
    for index, (example, probe, arm) in enumerate(batch):
        start = index * len(LABEL_ASSIGNMENTS)
        end = start + len(LABEL_ASSIGNMENTS)
        results.append(aggregate_counterbalanced_scores(
            example=example, probe=probe, label_assignments=LABEL_ASSIGNMENTS,
            permutation_scores=flat_scores[start:end], **arm,
        ))
    if batch_start % (20 * AGGREGATES_PER_BATCH) == 0:
        print(f'{min(batch_start + len(batch), len(aggregate_jobs)):,}/{len(aggregate_jobs):,}')

control_messages = [
    messages_for_elicitation(control, assignment)
    for control in elicitation_controls
    for assignment in LABEL_ASSIGNMENTS
]
control_scores = score_labels_batch(
    model=model, processor=processor, device=DEVICE, messages_batch=control_messages,
    choice_token_ids=CHOICE_TOKEN_IDS, enable_thinking=False,
)
elicitation_results = []
for index, control in enumerate(elicitation_controls):
    start = index * len(LABEL_ASSIGNMENTS)
    elicitation_results.append(aggregate_elicitation_scores(
        control=control, label_assignments=LABEL_ASSIGNMENTS,
        permutation_scores=control_scores[start:start + len(LABEL_ASSIGNMENTS)],
        enable_thinking=False,
    ))
write_jsonl(ARTIFACT_DIR / 'qwen_results.jsonl', results)
write_jsonl(ARTIFACT_DIR / 'qwen_elicitation_results.jsonl', elicitation_results)
print(f'Saved {len(results):,} aggregate decisions and {len(elicitation_results)} controls.')

In [ ]:
# Optional capability upper bound: allow Qwen to deliberate, then parse its final label.
# This generation metric is not comparable to the direct next-token decision logit.
RUN_THINKING_ARM = os.environ.get('RUN_THINKING_ARM', '0') == '1'
thinking_results = []
if RUN_THINKING_ARM:
    thinking_examples = [example for example in ladder_examples if example.stage == 'one_observation']
    thinking_jobs = [
        (example, probe, assignment)
        for example in thinking_examples
        for probe in example.probes
        for assignment in LABEL_ASSIGNMENTS
    ]
    generation_batch_size = int(os.environ.get('GENERATION_BATCH_SIZE', '12'))
    for start in range(0, len(thinking_jobs), generation_batch_size):
        batch = thinking_jobs[start:start + generation_batch_size]
        generated = generate_responses_batch(
            model=model, processor=processor, device=DEVICE, enable_thinking=True,
            messages_batch=[
                messages_for_probe(example, probe, assignment)
                for example, probe, assignment in batch
            ],
        )
        for (example, probe, assignment), output in zip(batch, generated):
            label_to_semantic = dict(zip(('A', 'B', 'C'), assignment))
            parsed_label = output['parsed_label']
            predicted = label_to_semantic.get(parsed_label)
            thinking_results.append({
                'example_id': example.example_id, 'bank_id': example.bank_id,
                'reliability': float(example.reliability), 'probe_id': probe.probe_id,
                'semantic_by_label': label_to_semantic,
                'normative_choice': probe.normative_choice,
                'predicted_semantic_choice': predicted,
                'correct': predicted == probe.normative_choice, **output,
            })
    write_jsonl(ARTIFACT_DIR / 'qwen_thinking_capability_results.jsonl', thinking_results)
    print(f'Saved {len(thinking_results)} thinking-enabled generated responses.')

## Result-record glossary

Each saved row represents one example/probe after aggregating all six label mappings. The most important fields are:

- `probe.normative_choice`: the exact Bayesian class—`left`, `right`, or `tie`.
- `probe.target_log_odds`: exact `log(P(left | evidence) / P(right | evidence))`. Positive favors left; negative favors right; zero is a tie.
- `behavioral_log_odds`: Qwen's average left-versus-right decision logit after mapping labels back to meaning. This is a preference score, **not an elicited posterior log-odds estimate**.
- `predicted_semantic_choice` and `counterbalanced_correct`: the aggregate three-way forced choice and whether it matches the exact target.
- `mapping_accuracy`: the fraction of the six individual label mappings that are correct. A gap between this and aggregate correctness indicates presentation sensitivity.
- `greedy_choice_compliance`: the fraction of mappings whose unrestricted first token is actually A, B, or C.
- `mean_choice_probability_mass`: the full-vocabulary probability assigned to A/B/C in total. Low mass warns that conditional forced-choice probabilities are off-distribution.
- `heuristic_predictions`: preregistered surface baselines such as mention counts and YES counts. Qwen should outperform these before its behavior is described as posterior computation.

The independent sampling unit is the generated schedule/world, not the six mappings and not the paired forward/reverse wording. Confidence intervals therefore resample schedules/worlds as clusters.

## Primary analysis

The summaries below never count label permutations as independent samples. Accuracy is shown with a schedule-clustered interval, class-balanced accuracy, exact-log-odds rank correlation, surface-heuristic baselines, option compliance, and orientation antisymmetry. The r=0.5 null is reported separately because all posterior comparisons tie.

## A practical order for interpreting the results

Do not start with overall accuracy. Read the diagnostics in this order:

1. **Elicitation control:** can the model compare explicit weights and handle an explicit equality? If not, the label interface is not validated.
2. **Greedy compliance and A/B/C mass:** is the conditional score describing an answer the model was actually willing to give?
3. **One-observation reliability inversion:** for identical evidence, does the left/right preference reverse between `r=0.1` and `r=0.9`? This is the cleanest test that the declared likelihood is used.
4. **Non-neutral ties:** does the model recognize equal accumulated evidence when `r` is not 0.5? This is harder and more diagnostic than the verbal `r=0.5` shortcut.
5. **Accumulation and composition:** after the minimal gate passes, does behavioral log-odds track exact log-odds as turns and candidates increase?
6. **Heuristic comparison and held-out worlds:** does the association survive relabeling and beat simple surface rules?

A high score at `r=0.5` alone is weak evidence because the prompt explicitly describes a 50%-reliable channel and every target is a tie. Conversely, failure at large `N` with near-zero A/B/C mass mainly diagnoses answer elicitation. The strongest evidence for Bayesian use would be correct paired reversals plus graded log-odds tracking on non-neutral conditions.

In [ ]:
if 'results' not in globals():
    with (ARTIFACT_DIR / 'qwen_results.jsonl').open(encoding='utf-8') as handle:
        results = [json.loads(line) for line in handle if line.strip()]

primary = [
    row for row in results
    if row['transcript_format'] == 'user_only'
    and row['answer_vocabulary'] == 'yes_no'
    and row['reliability_format'] == 'decimal'
    and not row['enable_thinking']
]
groups = defaultdict(list)
for row in primary:
    groups[(row['stage'], row['probe']['kind'], row['reliability'])].append(row)

summary = []
for key, rows in sorted(groups.items(), key=lambda item: str(item[0])):
    estimate, lower, upper = clustered_bootstrap_mean(
        rows, value=lambda row: float(row['counterbalanced_correct']),
        cluster=lambda row: (row['stage'], row['schedule_id'], row['world_id']),
    )
    finite = [row for row in rows if row['probe']['target_log_odds_state'] == 'finite']
    rho = spearman_correlation(
        [row['probe']['target_log_odds'] for row in finite],
        [row['behavioral_log_odds'] for row in finite],
    ) if len(finite) > 1 else math.nan
    summary.append({
        'group': key, 'n_decisions': len(rows),
        'independent_schedules': len({(row['schedule_id'], row['world_id']) for row in rows}),
        'accuracy': estimate, 'cluster_ci': (lower, upper),
        'balanced_accuracy': balanced_accuracy(
            rows, target=lambda row: row['probe']['normative_choice'],
            prediction=lambda row: row['predicted_semantic_choice'],
        ),
        'spearman_log_odds': rho,
        'choice_compliance': sum(row['greedy_choice_compliance'] for row in rows) / len(rows),
    })
for row in summary:
    print(row)

print('Candidate heuristic baselines:', heuristic_accuracies(primary))

robustness_groups = defaultdict(list)
for row in results:
    arm = (row['transcript_format'], row['answer_vocabulary'], row['reliability_format'])
    if arm != ('user_only', 'yes_no', 'decimal'):
        robustness_groups[arm].append(row)
for arm, rows in robustness_groups.items():
    print('robustness-arm', arm, {
        'n': len(rows),
        'accuracy': sum(row['counterbalanced_correct'] for row in rows) / len(rows),
        'balanced_accuracy': balanced_accuracy(
            rows, target=lambda row: row['probe']['normative_choice'],
            prediction=lambda row: row['predicted_semantic_choice'],
        ),
    })

n8_primary = [row for row in primary if row['stage'] == 'n8']
natural_groups = defaultdict(list)
for row in n8_primary:
    natural_groups[(row['probe']['kind'], row['reliability'])].append(row)
for key, rows in sorted(natural_groups.items(), key=str):
    weighted_correct = sum(
        row['prior_predictive_probability'] * float(row['counterbalanced_correct'])
        for row in rows
    )
    weight = sum(row['prior_predictive_probability'] for row in rows)
    print('natural-weighted', key, weighted_correct / weight)

def fit_line(rows):
    xs = [row['probe']['target_log_odds'] for row in rows]
    ys = [row['behavioral_log_odds'] for row in rows]
    x_mean, y_mean = sum(xs) / len(xs), sum(ys) / len(ys)
    denominator = sum((x - x_mean) ** 2 for x in xs)
    slope = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys)) / denominator
    return y_mean - slope * x_mean, slope

for stage in sorted({row['stage'] for row in primary}):
    finite = [
        row for row in primary
        if row['stage'] == stage and row['probe']['kind'] == 'candidate'
        and row['probe']['target_log_odds_state'] == 'finite'
    ]
    cluster_number = lambda row: row['schedule_id'] if stage == 'n8' else row['world_id']
    train = [row for row in finite if cluster_number(row) % 2 == 0]
    test = [row for row in finite if cluster_number(row) % 2 == 1]
    if train and test:
        intercept, slope = fit_line(train)
        rho = spearman_correlation(
            [row['probe']['target_log_odds'] for row in test],
            [row['behavioral_log_odds'] for row in test],
        )
        mse = sum(
            (row['behavioral_log_odds'] - (intercept + slope * row['probe']['target_log_odds'])) ** 2
            for row in test
        ) / len(test)
        print('held-out-log-odds', stage, {'slope': slope, 'rho': rho, 'mse': mse})

In [ ]:
finite_candidate_rows = [
    row for row in primary
    if row['probe']['kind'] == 'candidate'
    and row['probe']['target_log_odds_state'] == 'finite'
]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for stage in sorted({row['stage'] for row in finite_candidate_rows}):
    stage_rows = [row for row in finite_candidate_rows if row['stage'] == stage]
    axes[0].scatter(
        [row['probe']['target_log_odds'] for row in stage_rows],
        [row['behavioral_log_odds'] for row in stage_rows],
        s=12, alpha=0.25, label=stage,
    )
axes[0].axhline(0, color='gray', linewidth=1)
axes[0].axvline(0, color='gray', linewidth=1)
axes[0].set(xlabel='Exact posterior log-odds', ylabel='Behavioral decision logit')
axes[0].legend()

by_orientation_pair = defaultdict(dict)
for row in primary:
    probe_id = row['probe']['probe_id']
    orientation = 'forward' if probe_id.endswith('_forward') else 'reverse'
    pair_id = probe_id.removesuffix('_forward').removesuffix('_reverse')
    by_orientation_pair[(row['example_id'], pair_id)][orientation] = row
antisymmetry_errors = [
    abs(pair['forward']['behavioral_log_odds'] + pair['reverse']['behavioral_log_odds'])
    for pair in by_orientation_pair.values()
    if set(pair) == {'forward', 'reverse'}
]
axes[1].hist(antisymmetry_errors, bins=40)
axes[1].set(xlabel='|forward logit + reversed logit|', ylabel='Paired probe count')
fig.suptitle('Posterior structure and orientation control')
fig.tight_layout()
plt.show()
print('Median orientation antisymmetry error:', sorted(antisymmetry_errors)[len(antisymmetry_errors)//2])

## Behavioral stopping point

This notebook intentionally stops after behavioral validation. It does not capture activations, train layer-wise probes, localize circuits, or perform activation patching. First establish: (1) Stage-0 label/equality competence; (2) paired ranking reversal between r=0.1 and r=0.9 in the one-observation task; (3) positive held-out association between behavioral and exact log-odds across world relabelings; and (4) performance beyond the preregistered heuristics. Mechanistic experiments should be designed later, conditional on those results.